<h1 style="text-align:center;">
Amazon delivery - EDA
</h1>

# 1. Problem definition
### The project begins with an exploratory data analysis of the dataset to understand its structure, data quality, and the relationships between key variables. Through this analysis, the study aims to identify patterns, trends, and potential correlations within the data related to delivery operations. The goal of this exploratory phase is to gain a clearer understanding of the dataset and the factors associated with delivery time before applying any further analytical or predictive methods.

### https://www.kaggle.com/datasets/sujalsuthar/amazon-delivery-dataset/data

# 2. Data Loading

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from geopy.distance import geodesic

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 3. Data cleaning

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/sujalsuthar/amazon-delivery-dataset/amazon_delivery.csv')
df_raw = df.copy()
df

In [ ]:
df.describe(include='all')

## Data quality table

In [ ]:
# Data quality table
data_quality = pd.DataFrame({
    "dtype": df.dtypes,
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
    "unique_values": df.nunique()  # valori unice care exista in tabel
})

# Flags
data_quality["is_constant"] = data_quality["unique_values"] <= 1
data_quality["is_high_missing"] = data_quality["missing_percent"] > 30
data_quality["is_duplicate_col"] = df.T.duplicated().values

# Format percent
data_quality["missing_percent"] = data_quality["missing_percent"].astype(str) + "%"

data_quality

In [ ]:
cat_cols = ["Weather", "Traffic", "Vehicle", "Area", "Category"]

# vezi valorile unice
for col in cat_cols:
    print(col)
    print(df[col].unique())
    print()

# curățare spații
df[cat_cols] = df[cat_cols].apply(lambda x: x.astype("string").str.strip())

# transformăm toate variantele de nan în NA
df[cat_cols] = df[cat_cols].replace(["NaN", "nan", "NAN", "None", "null"], pd.NA)

# înlocuim valorile lipsă
df[cat_cols] = df[cat_cols].fillna("Missing")

In [ ]:
df["Traffic"].value_counts(dropna=False)

In [ ]:
def calculate_distance(row):
    start = (row['Store_Latitude'], row['Store_Longitude'])
    end = (row['Drop_Latitude'], row['Drop_Longitude'])
    return geodesic(start, end).km
df.loc[:, 'Distance'] = df.apply(calculate_distance, axis=1)
# Distance in km

In [ ]:
# Eliminate unuseful columns
df = df.drop(columns=[
    'Store_Latitude',
    'Store_Longitude',
    'Drop_Latitude',
    'Drop_Longitude'
])
df.head()

In [ ]:
# What time period does the data cover?
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print(df['Order_Date'].min())
print(df['Order_Date'].max())

In [ ]:
# calculate pickup time

# curățare + conversie
df['Pickup_Time'] = pd.to_timedelta(df['Pickup_Time'].astype(str).str.strip(), errors='coerce')
df['Order_Time'] = pd.to_timedelta(df['Order_Time'].astype(str).str.strip(), errors='coerce')

# diferența
diff = df['Pickup_Time'] - df['Order_Time']

# corecție pentru trecerea la ziua următoare
diff = diff.mask(diff < pd.Timedelta(0), diff + pd.Timedelta(days=1))

# minute
df['pickup_relative_time'] = diff.dt.total_seconds() / 60
df.head()

In [ ]:
# Eliminate unuseful columns
df = df.drop(columns=[
    'Order_Date',
    'Order_Time',
    'Pickup_Time',
    'Order_ID'
])
df.head()

# 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))

sns.histplot(df["Delivery_Time"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Delivery Time (min)")

sns.histplot(df["Distance"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Distribution of Distance (km)")

plt.tight_layout()
plt.show()

In [ ]:
bins = pd.cut(df["Distance"], bins=10)

distance_distribution = pd.DataFrame({
    "Count": bins.value_counts().sort_index(),
    "Percent": bins.value_counts(normalize=True).sort_index()*100
})

print(distance_distribution)

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))

sns.scatterplot(x="Distance", y="Delivery_Time", data=df, ax=axes[0])
axes[0].set_title("Distance vs Delivery Time")

sns.scatterplot(x="Agent_Age", y="Delivery_Time", data=df, ax=axes[1])
axes[1].set_title("Agent Age vs Delivery Time")

sns.scatterplot(x="Agent_Rating", y="Delivery_Time", data=df, ax=axes[2])
axes[2].set_title("Agent Rating vs Delivery Time")

plt.tight_layout()
plt.show()

### First of all, the dataset contains inconsistencies in the latitude and longitude values. Some calculated distances reach thousands of kilometers, which is not realistic for delivery operations within the given time frame. It is therefore likely that certain coordinate values are incorrect or represent outliers in the data. Additionally, the variables Agent Age and Agent Rating appear to have little influence on the delivery time and may be less relevant for explaining variations in delivery performance.

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))

sns.boxplot(x="Traffic", y="Delivery_Time", data=df, ax=axes[0])
axes[0].set_title("Traffic vs Delivery Time")

sns.boxplot(x="Weather", y="Delivery_Time", data=df, ax=axes[1])
axes[1].set_title("Weather vs Delivery Time")

sns.boxplot(x="Vehicle", y="Delivery_Time", data=df, ax=axes[2])
axes[2].set_title("Vehicle vs Delivery Time")

plt.tight_layout()
plt.show()

It appears that, regardless of the vehicle type, the delivery time remains within a similar range.

### Next: What"s the correlation between difeferent columns and categories?

In [ ]:
# coloane numerice
num_cols = [
    "Agent_Age",
    "Agent_Rating",
    "Distance",
    "pickup_relative_time",
    "Delivery_Time"
]

# coloane categorice relevante
cat_cols = [
    "Weather",
    "Traffic",
    "Vehicle",
    "Area",
    "Category"
]

# encoder sklearn
encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(df[cat_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(cat_cols)
)

# combinăm datele
data = pd.concat([df[num_cols].reset_index(drop=True), encoded_df], axis=1)

corr_delivery = data.corr()[["Delivery_Time"]].sort_values(by="Delivery_Time", ascending=False)

plt.figure(figsize=(6,10))
sns.heatmap(corr_delivery, annot=True, cmap="coolwarm")
plt.title("Correlation with Delivery Time")
plt.show()

In [ ]:
pivot = df.pivot_table(
    values="Delivery_Time",
    index="Vehicle",
    columns="Traffic",
    aggfunc="mean"
)

print(pivot)

In [ ]:
sns.heatmap(pivot, annot=True, cmap="coolwarm",fmt=".1f")
plt.title("Average Delivery Time by Vehicle and Traffic")
plt.show()

In [ ]:
pivot = df.pivot_table(
    values="Delivery_Time",
    index="Vehicle",
    columns="Weather",
    aggfunc="mean"
)

print(pivot)

In [ ]:
sns.heatmap(pivot, annot=True, cmap="coolwarm",fmt=".1f")
plt.title("Average Delivery Time by Vehicle and Traffic")
plt.show()

### How can be deen the most influencial factor is Traffic

# 5. Conclusions

### There are several factors that influence delivery time. The most important factor is distance. However, some inconsistencies were identified in the latitude and longitude data, which produce unrealistic distances and suggest the presence of outliers or incorrect geographic coordinates.

### Another important factor is traffic conditions. Heavy or jammed traffic can significantly slow down deliveries, even for motorcycles, which are generally more flexible in congested urban environments.

### There are also some concerns regarding the authenticity and reliability of certain parts of the dataset. The analysis suggests that delivery times appear relatively similar across different vehicle types, which raises questions about whether the vehicle variable has a strong impact on delivery performance in this dataset.
